# Which links get QKD? Greedy vs exact vs QAOA

*Secure the Grid · Step 3 stretch · classical vs hybrid planning*

We already decided every link gets post-quantum crypto. **QKD is scarce** (cost bands in notebook 5). Under a budget, which fibers should get tap detection?

- **Value:** criticality, critical loads (hospital / water / airport), backbone / regional ties  
- **Cost:** mid-band QKD units (equipment + dedicated fiber)  
- **Methods:** greedy by value/cost, exact search (2¹⁶), and **QAOA** on the local Aer simulator (same pattern as the TVA QAOA primer)

Honest expectation at 16 links: classical exact/greedy should win or tie; QAOA is the hybrid check.

In [ ]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

from pathlib import Path
import sys
_here = Path.cwd().resolve()
for _cand in (_here, _here.parent):
    if (_cand / "secure_grid").is_dir():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

import pandas as pd
from secure_grid import qkd_select as qs
from secure_grid import attacks

cands = qs.candidates()
print(pd.DataFrame(cands)[["id", "role", "criticality", "value", "cost"]].round(2).to_string(index=False))

## Run the three solvers

Default budget = 4 × cost of one metro QKD link (same scale as our hand-picked design set).

In [ ]:
import time
t0 = time.time()
# p=1 is faster; set p=2 for a bit more QAOA depth
out = qs.run(budget=None, p=1)
print(f"Budget: {out['budget']:.0f} units (one link ≈ {out['unit_cost_mid_link']:.0f})")
for name in ("greedy", "exact", "qaoa"):
    r = out[name]
    print(f"\n{name}: value={r['value']:.1f}  cost={r['cost']:.0f}  n={r['n']}")
    print("  links:", ", ".join(r["links"]))
print("\nMatch exact?", out["match_exact"])
print(f"Runtime: {time.time() - t0:.1f}s")

hand = attacks.load_defenses()["design"]["qkd_links"]
print("\nOur hand-picked design set:", ", ".join(hand))
print("Same as exact?", set(hand) == set(out["exact"]["links"]))

## What this means

On this metro network, **exact and greedy agree**, and they match the four links we had already chosen (backbone + hospital + water + airport). QAOA returns a feasible set with similar value (sometimes the same, sometimes swapping one high-value link). That is the honest result: at this size you do not need a quantum optimizer to place four QKD boxes — but we showed the hybrid pipeline works and can scale when the city model grows past brute force.

**Optional (qBraid Lab):** rebuild the optimized QAOA circuit and submit to a free simulator with `secure_grid.runs` (`max_credits=0`). The design decision itself is already fixed by the comparison above.

In [ ]:
# Optional free qBraid submit — off by default
RUN_QBRAID = False
if RUN_QBRAID:
    from secure_grid import runs
    from secure_grid.qkd_select import qubo_matrices, qaoa_circuit, candidates
    import numpy as np
    cands = candidates()
    budget = 4 * cands[0]["cost"]
    linear, quad, _ = qubo_matrices(cands, budget)
    # shallow circuit for free backends
    params = np.array([0.8, 0.6])  # one layer gamma, beta (replace with optimized if you re-run)
    qc = qaoa_circuit(params, linear, quad, p=1)
    job = runs.submit("qbraid:qbraid:sim:qir-sv", qc, shots=200,
                      purpose="QAOA QKD link selection (16 qubits, p=1)")
    print("job", job.id)
else:
    print("Set RUN_QBRAID = True in qBraid Lab to log a free simulator job.")